# Chapter 2: Data Ingestion & Variables - Exercise

In [2]:
low_memory=False
import numpy as np
import pandas as pd

## 2.1 Introduction and Problem Setting

Today you begin work on a practical fraud detection exercise.

A neighbouring company has reported a new type of credit card fraud. The exact cause is unknown but the fraudster can create multiple transactions at the same time, which may result in the same amount being recorded more than once. Your task is to check whether our company is affected and, if so, identify any client linked to suspicious transactions.

You receive three datasets from our offices:

- London (all transactions in UTC time): `Transactions_UTC_upload.txt`
- Chicago (US Central time): `Transactions_US_Central_upload.txt`
- Shanghai: `Transactions_China_upload.txt`

Client details are in `Credit_customers.txt`.

If you find suspicious activity, report the client name(s) associated with those transactions.

## 2.2 Data ingestion and exploration

First, load each dataset into its own `pandas` DataFrame. Inspect the first rows, the column names, and the column types to get a quick overview.

Notes about `pd.read_csv()` options you will see in the code below:

- `sep` sets the field delimiter used in the file. Common values are `','` for comma separated files and `';'` or `'\t'` for semicolon or tab separated files. Use the correct `sep` so `pandas` splits columns properly.
- `index_col=0` tells `pandas` to use the first column in the file as the DataFrame index (row labels) rather than the default numeric row numbers. This is useful when the file already contains an ID or name column you want to use to reference rows.

Example: `pd.read_csv('Credit_customers.txt', sep='\t', index_col=0)` reads a tab separated file and uses the first column as the row index.

In [5]:
pd.read_csv("Transactions_China_upload.txt",sep=';',index_col=0)

,Transaction_time,time_zone
Client_ID,,
805,2019-12-15 03:37:36,China
67,2019-12-15 06:42:35,China
749,2019-12-15 14:48:02,China
828,2019-12-15 22:17:06,China
539,2019-12-15 23:30:35,China
...,...,...
675,2022-02-20 08:36:29,China
718,2022-02-20 12:22:28,China
703,2022-02-20 17:07:02,China


In [6]:
pd.read_csv("Transactions_UTC_upload.txt",sep=';',index_col=0)

,Transaction_time,time_zone
Client_ID,,
86,2019-12-15 13:39:21,UTC
313,2019-12-16 18:00:26,UTC
198,2019-12-17 06:20:07,UTC
466,2019-12-17 06:47:42,UTC
119,2019-12-17 11:46:27,UTC
...,...,...
393,2022-02-21 01:16:20,UTC
801,2022-02-21 04:39:50,UTC
233,2022-02-21 21:55:32,UTC


In [7]:
pd.read_csv("Transactions_US_Central_upload.txt",sep=';',index_col=0)

,Transaction_time,time_zone
Client_ID,,
148,2019-12-15 05:24:35,US_Central
299,2019-12-15 13:09:57,US_Central
197,2019-12-15 13:30:02,US_Central
259,2019-12-15 14:26:03,US_Central
591,2019-12-16 06:31:55,US_Central
...,...,...
793,2022-02-19 16:27:46,US_Central
446,2022-02-19 22:03:07,US_Central
292,2022-02-20 11:26:17,US_Central


In [9]:
pd.read_csv("Credit_customers.txt",sep='\t',index_col=0)

,Name
Client_ID,
1,"Braund, Mr. Owen Harris"
2,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
3,"Heikkinen, Miss. Laina"
4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
5,"Allen, Mr. William Henry"
...,...
887,"Montvila, Rev. Juozas"
888,"Graham, Miss. Margaret Edith"
889,"Johnston, Miss. Catherine Helen \Carrie\"""""


## 2.3 Data Processing

After loading the files, convert the transaction timestamp columns to `datetime` objects so you can compare times across sources. When data comes from different time zones, localize each timestamp to its original zone and then convert to a common timezone such as UTC before comparing.

Now look for duplicate timestamps that may indicate multiple simultaneous transactions. If you are unsure how to proceed, search for common techniques to compare datasets or to find duplicate timestamps. Try to reason about timezone differences and whether identical UTC timestamps could indicate suspicious behavior.

In [18]:
df = pd.read_csv("Transactions_US_Central_upload.txt",sep=";",index_col=0)
df['Transaction_time']=pd.to_datetime(df['Transaction_time'])
df1 = pd.read_csv("Transactions_China_upload.txt",sep=";",index_col=0)
df1['Transaction_time']=pd.to_datetime(df1['Transaction_time'])
df2 = pd.read_csv("Transactions_UTC_upload.txt",sep=";",index_col=0)
df2['Transaction_time']=pd.to_datetime(df2['Transaction_time'])

In [19]:
print(df.head(10))
print(df1.head(10))
print(df2.head(10))

             Transaction_time   time_zone
Client_ID                                
148       2019-12-15 05:24:35  US_Central
299       2019-12-15 13:09:57  US_Central
197       2019-12-15 13:30:02  US_Central
259       2019-12-15 14:26:03  US_Central
591       2019-12-16 06:31:55  US_Central
343       2019-12-16 06:33:04  US_Central
184       2019-12-16 11:54:35  US_Central
300       2019-12-16 18:15:15  US_Central
736       2019-12-16 22:52:28  US_Central
423       2019-12-17 00:21:34  US_Central
             Transaction_time time_zone
Client_ID                              
805       2019-12-15 03:37:36     China
67        2019-12-15 06:42:35     China
749       2019-12-15 14:48:02     China
828       2019-12-15 22:17:06     China
539       2019-12-15 23:30:35     China
408       2019-12-16 09:00:11     China
870       2019-12-16 20:36:00     China
484       2019-12-17 01:55:01     China
286       2019-12-17 03:15:43     China
234       2019-12-17 05:08:10     China
             Tra

If you find duplicate transaction timestamps, try to link those rows to the client table using the customer ID to see whether a known client appears in the suspicious records.